# Stream Analysis — Overview Pipeline

This notebook runs Part 1 of the document image analysis pipeline:

1. Extract visual embeddings (DINOv2 or CLIP)
2. Extract layout features
3. Cluster with UMAP + HDBSCAN
4. Tag a sample with a VLM (optional — requires `ANTHROPIC_API_KEY`)
5. Generate visualisations

All heavy computations are cached to `output_dir`, so re-running cells
after the first run is fast.

This pipeline works on a plain directory of images -- it has no dependency on PageXML or
any particular archive's layout. For a runnable demo it points at the same `NL-AsnDA_0114.11_1`
thumbnails (630 scans) the other demo notebooks use, but point `cfg.image_dir` at any directory
of document images to run it on your own collection.

In [1]:
import os
os.environ.setdefault('TQDM_DISABLE', '1')  # keep notebook outputs tidy -- progress bars don't add value once baked in

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s  %(levelname)-8s  %(message)s', datefmt='%H:%M:%S')

## Configuration

Edit the values below, or load from a YAML file with `AnalysisConfig.from_yaml('path/to/config.yaml')`.

In [2]:
from archival_structures.stream_analysis import AnalysisConfig

cfg = AnalysisConfig(
    image_dir='../../data/thumbs/NL-AsnDA/NL-AsnDA_0114.11/NL-AsnDA_0114.11_1',
    output_dir='../../archival_structures/stream_analysis/outputs/demo',
    embedding_model='dinov2',
    batch_size=32,
    hdbscan_min_cluster_size=10,
    vlm_sample_size=100,
)

# Or load from YAML:
# cfg = AnalysisConfig.from_yaml('archival_structures/stream_analysis/config.yaml')

print(cfg)

AnalysisConfig(image_dir='../../data/thumbs/NL-AsnDA/NL-AsnDA_0114.11/NL-AsnDA_0114.11_1', image_extensions=['jp2', 'jpg', 'jpeg', 'png', 'tif', 'tiff'], batch_size=32, output_dir='../../archival_structures/stream_analysis/outputs/demo', embedding_model='dinov2', clip_model='openai/clip-vit-base-patch32', dinov2_model='facebook/dinov2-base', combine_layout_features=True, layout_weight=0.2, umap_n_neighbors=15, umap_min_dist=0.1, umap_n_components=2, umap_metric='cosine', umap_random_state=42, hdbscan_min_cluster_size=10, hdbscan_min_samples=5, hdbscan_metric='euclidean', vlm_provider='anthropic', vlm_model='claude-haiku-4-5', vlm_sample_size=100, vlm_batch_size=1, cluster_grid_n=9, umap_point_size=3, umap_alpha=0.6, samples_per_cluster=30, active_learning_batch_size=20, active_learning_model='logistic_regression', uncertainty_strategy='margin')


## Step 1 & 2: Embeddings + Layout Features

In [3]:
from archival_structures.stream_analysis.overview.embeddings import extract_embeddings
from archival_structures.stream_analysis.overview.layout_analysis import extract_all_layout_features

config = cfg.to_dict()

embeddings, image_ids = extract_embeddings(config)
print(f'Embeddings: {embeddings.shape}  ({len(image_ids)} images)')

layout_features = extract_all_layout_features(config)
print(f'Layout features: {len(layout_features)} images')

16:30:25  INFO      Using device: mps


16:30:25  INFO      Loading DINOv2 model: facebook/dinov2-base


16:30:25  INFO      HTTP Request: HEAD https://huggingface.co/facebook/dinov2-base/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"


16:30:25  WARNING   Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


16:30:25  INFO      HTTP Request: HEAD https://huggingface.co/facebook/dinov2-base/resolve/main/preprocessor_config.json "HTTP/1.1 307 Temporary Redirect"


...

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

16:30:42  INFO      Extracted and cached 630 embeddings of dimension 768


Embeddings: (630, 768)  (630 images)


16:30:46  INFO      Extracted layout features for 630 images → ../../archival_structures/stream_analysis/outputs/demo/layout_features.json


Layout features: 630 images


## Step 3: Clustering

In [4]:
from archival_structures.stream_analysis.overview.clustering import run_clustering, get_cluster_members

clustering = run_clustering(embeddings, image_ids, config, layout_features)
cluster_members = get_cluster_members(clustering)

print(f"Clusters: {clustering['n_clusters']}  |  Outliers: {clustering['n_outliers']}")
for c_id, members in sorted(cluster_members.items()):
    name = 'outliers' if c_id == -1 else f'Cluster {c_id}'
    print(f'  {name}: {len(members)} images')

16:30:46  INFO      Combining embeddings with layout features


16:30:46  INFO      Running UMAP (n_neighbors=15, metric=cosine) on 630 images …


/opt/homebrew/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


16:30:51  INFO      UMAP projection cached → ../../archival_structures/stream_analysis/outputs/demo/umap_projection.json


16:30:51  INFO      Running HDBSCAN (min_cluster_size=10) …


...

16:30:51  INFO        Cluster 6: 214 images


16:30:51  INFO        Cluster 7: 257 images


16:30:51  INFO      Clustering results cached → ../../archival_structures/stream_analysis/outputs/demo/clustering_results.json


16:30:51  INFO      Cluster hierarchy cached → ../../archival_structures/stream_analysis/outputs/demo/cluster_hierarchy.npz


Clusters: 8  |  Outliers: 11
  outliers: 11 images
  Cluster 0: 29 images
  Cluster 1: 26 images
  Cluster 2: 40 images
  Cluster 3: 11 images
  Cluster 4: 17 images
  Cluster 5: 25 images
  Cluster 6: 214 images
  Cluster 7: 257 images


## Step 4: VLM Tagging (optional)

Requires `ANTHROPIC_API_KEY` to be set in the environment. Skip this cell if you don't have one.

In [5]:
import os
vlm_tags = {}

if os.getenv('ANTHROPIC_API_KEY'):
    from archival_structures.stream_analysis.overview.vlm_tagging import run_vlm_tagging, summarise_vlm_tags
    vlm_tags = run_vlm_tagging(config, image_ids)
    summary = summarise_vlm_tags(vlm_tags)
    for field, counts in summary.items():
        print(f'{field}: {counts}')
else:
    print('ANTHROPIC_API_KEY not set — skipping VLM tagging.')

ANTHROPIC_API_KEY not set — skipping VLM tagging.


## Step 5: Visualisations

In [6]:
from archival_structures.stream_analysis.overview.visualization import (
    plot_umap,
    plot_cluster_grids,
    plot_layout_distributions,
    plot_vlm_summary,
)

plot_umap(clustering, config)
plot_cluster_grids(clustering, config)
plot_layout_distributions(clustering, layout_features, config)
if vlm_tags:
    plot_vlm_summary(vlm_tags, config)

print(f"Visualisations saved to {cfg.visualization_dir}/")

16:30:51  INFO      UMAP plot saved → ../../archival_structures/stream_analysis/outputs/demo/visualizations/umap_clusters.png


16:30:52  INFO      Cluster grids saved to ../../archival_structures/stream_analysis/outputs/demo/visualizations


16:30:53  INFO      Layout distribution plot saved → ../../archival_structures/stream_analysis/outputs/demo/visualizations/layout_distributions.png


Visualisations saved to ../../archival_structures/stream_analysis/outputs/demo/visualizations/


## Or: run the full pipeline in one call

In [7]:
from archival_structures.stream_analysis import run_overview

clustering = run_overview(cfg, run_vlm=False)  # set run_vlm=True if you have an API key

16:30:53  INFO      === Step 1: Extracting visual embeddings ===


16:30:53  INFO      Loading cached embeddings from disk


16:30:53  INFO      Loaded 630 embeddings of dimension 768


16:30:53  INFO        630 images, embedding dim = 768


16:30:53  INFO      === Step 2: Extracting layout features ===


...

16:30:53  INFO      UMAP plot saved → ../../archival_structures/stream_analysis/outputs/demo/visualizations/umap_clusters.png


16:30:53  INFO      Cluster grids saved to ../../archival_structures/stream_analysis/outputs/demo/visualizations


16:30:54  INFO      Layout distribution plot saved → ../../archival_structures/stream_analysis/outputs/demo/visualizations/layout_distributions.png


16:30:54  INFO      All visualisations saved to ../../archival_structures/stream_analysis/outputs/demo/visualizations/


16:30:54  INFO      Overview pipeline complete. Review the cluster grids and UMAP plot.
